In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

plt.style.use("ggplot")

%matplotlib inline

In [2]:
df = pd.read_csv(
    "../data/MachineLearningRating_v3.txt",
    sep="|",
    low_memory=False
)

df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [3]:
df["LossRatio"] = df["TotalClaims"] / df["TotalPremium"]

df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

df["HasClaim"] = np.where(df["TotalClaims"] > 0, 1, 0)

df[["LossRatio", "Margin", "HasClaim"]].head()

,LossRatio,Margin,HasClaim
0,0.0,21.929825,0
1,0.0,21.929825,0
2,NaN,0.000000,0
3,0.0,512.848070,0
4,NaN,0.000000,0


In [4]:
province_mean = df.groupby("Province")["TotalClaims"].mean()

province_mean.sort_values(ascending=False)

Province
KwaZulu-Natal    84.234293
Gauteng          74.630009
Western Cape     60.831482
Eastern Cape     44.713432
Free State       43.822975
North West       41.317426
Limpopo          40.927553
Mpumalanga       38.785147
Northern Cape    14.026726
Name: TotalClaims, dtype: float64

In [5]:
group_a = df[df["Province"] == "Gauteng"]["TotalClaims"]

group_b = df[df["Province"] == "Western Cape"]["TotalClaims"]

stat, p_value = ttest_ind(
    group_a,
    group_b,
    nan_policy='omit'
)

print("P-value:", p_value)

P-value: 0.05632044649871941


In [6]:
if p_value < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


In [7]:
male = df[df["Gender"] == "Male"]["TotalClaims"]

female = df[df["Gender"] == "Female"]["TotalClaims"]

stat, gender_p = ttest_ind(
    male,
    female,
    nan_policy='omit'
)

print("P-value:", gender_p)

P-value: 0.8041073961270342


In [8]:
zip_margin = df.groupby("PostalCode")["Margin"].mean()

zip_margin.sort_values().head()

PostalCode
466    -2104.003715
2920   -1613.177313
1342   -1511.886460
1751   -1221.090842
9756   -1125.351747
Name: Margin, dtype: float64

In [9]:
zip1 = df[df["PostalCode"] == 2000]["Margin"]

zip2 = df[df["PostalCode"] == 8000]["Margin"]

stat, zip_p = ttest_ind(
    zip1,
    zip2,
    nan_policy='omit'
)

print("P-value:", zip_p)

P-value: 0.7123247716247572


In [10]:
results = pd.DataFrame({
    "Hypothesis": [
        "Province Risk Difference",
        "Gender Risk Difference",
        "Zip Code Margin Difference"
    ],
    "P-Value": [
        p_value,
        gender_p,
        zip_p
    ]
})

results

,Hypothesis,P-Value
0,Province Risk Difference,0.056320
1,Gender Risk Difference,0.804107
2,Zip Code Margin Difference,0.712325


We reject H0 for provinces because the p-value is below 0.05.

This indicates significant risk differences between provinces.
ACIS can consider province-based premium adjustments.